> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# Conflict-v3： New values, new properties, new templates and simple  baseline（ supplemental edition) 

** Continue Current  runtime。 Copy the four code units of this document in turn to the original  v3 notebook  Implementation at the end. **
 This supplement contains all extensions, but restores the original  notebook  Loaded  `backend`、 Tools and identity information; it does not independently load models. 
 No upload required  `.py`， There is no need to re-installation or rerun the original experiment.  runtime  In directly running. 

 Reservation  Llama-3.1-8B base、 Fixed weights, bf16、TransformerLens  and 
`check_stock → reserve_item → create_order`。 No new one.  head search， nor read the original  test  or after selection  head。

##  Fixed Experimental Contents 

- 192  A new one.  source：3  attribute  × 4  New Template  × 2  Inventory status  × 8  repeating;each attribute  64  One. 
-  Properties As  `delivery_city`、`order_customer`、`preferred_item`， This affects the actual ordering city, accounts and commodities, respectively. 
- 12  A city that was not used in previous experiments, including  New York、Los Angeles、Cape Town、Buenos Aires  equals multiple words;does not press singles  token  Filter. 
-  Theres only one input left.  verified_profile  And one.  retrieved_note；aligned/conflict  Change Only  note.value。
-  each  source  Fully cross two record orders with three treatments: unchanged, verified records first, and verified records only. 
-  Two.  baseline  use source labels rather than scoring answers. They assume reliable labels, and the filtering baseline also reduces available information. 
-  Use excluding  primary/backup  Waiting for the answer.  SKU。 Tasks are given only in a visible, non-focus field, by which the focus is really derived  verified_profile  Provide. 

 This expands values, attributes, and wording within the local ordering task, not an external  benchmark， nor a new task domain. 
“ Unseen means unused in earlier experiments, not absent from model pretraining. The new input format differs from the old experiment, so results are not directly pooled. 

##  Cost and running 

 Every  source  Yes.  12  Logical comparison of lines, but only produced  5  A different valid input, so complete  evaluation  Run at most 
960   Actor episode  and corresponding  Writer。** One.  Actor episode  Include multiple model generation **， Not equal to a call. 
 Run for independence first.  6   smoke source， Every time.  24   evaluation source； Repeat the third block and you can go on. 
 Could also set  `max_sources=None`  Run all remaining tasks in one pass. Retain every condition, not only tasks where errors occur. 

 Saves the result in the original  RUN_DIR  At the same level  `_generalization_v1`  directory. Each generation and each complete  source  Checkpoints are available; 
 After ordinary interruption, rerun the same code to reuse results. Do not overwrite this directory after changing code or configuration. 

##  How to read the results 

Actor： Report full task success, strict tool-path success, focal-field errors in accepted orders, adoption of conflict values, format errors, tool errors, and no orders separately. 
 When the preferred commodity is missing, the correct final order is  fallback；Writer  The preference that should be retained remains  verified preferred_item。
Writer  From each  arm  Actual presentation of information and actual  Actor  Track Generate Full  JSON， And it separates the scoring values, the source, the combined correctness rate and the whole memory. 
Writer   baseline  Differences include input information and  Actor  The overall effect of the changed trajectory is not attributable separately to  Writer  Internal mechanisms. 
 This edition is not added  Writer  Post-output follow-up mission dissemination. 

 The main comparison is fixed as  untreated、note_first  Down  conflict−aligned  ;the accepted focus field error rate; 
 Baseline effects, location effects and attribute stratification are secondary analysis. CSV  Give Count / Independence source denominators, Wilson  Inter-segment, pairing differences, 
source bootstrap  Matching tests with two items that have not been multiple-corrected.  bootstrap  It would be clearly marked, and it would not be possible to assert that it had no effect. 
 Not full.  192  Show when  PARTIAL， Intermediate results are not a complete evaluation. Changes after repeated inspection of results are subsequent development. 

 All new models are still to be shown.  Colab  execution. Local validation checks genuine tool workflows with mocked generation, scoring, protocols, and checkpoint recovery. 


## 1 ·  Register extension and lock independence  manifest
 Copy the entire code unit below to the original  notebook  End. Use only existing  tokenizer  Check input length along normal paths without generating model answers; actual inputs still use the original  backend  Check, and do not stop silently. 


In [ ]:
# BLOCK 1: paste into the end of the ORIGINAL v3 notebook, in its current runtime.
# Self-contained: no script upload, installs, model reload, or training.
GX_EXTENSION_SOURCE = r'''
"""New-value, new-attribute order tasks using the original executable tools.

This append-only module needs the original OrderEnvironment, parse_json_output
and canonical_json. It never changes their definitions or the loaded backend.
Gold fields remain evaluator data: the Actor sees only its focal source records
and the other, fixed task fields. All reported outcomes come from real tool
transitions after free generation of complete actions.
"""
import copy as gx_copy
import random as gx_random
import time as gx_time


GX_ATTRIBUTES = ("delivery_city", "order_customer", "preferred_item")
GX_CITIES = (
    "Lisbon", "Prague", "Helsinki", "Warsaw", "Cairo", "Nairobi",
    "Mumbai", "Osaka", "New York", "Los Angeles", "Cape Town", "Buenos Aires",
)
GX_TEMPLATES = (
    ("gx_dispatch", "Handle request {request_id}: create one order using its resolved fields. "
     "Use its preferred item if stocked; otherwise use the listed fallback."),
    ("gx_shipment", "For request {request_id}, arrange exactly one shipment. Choose the requested "
     "preferred SKU when available, else the permitted fallback."),
    ("gx_inventory", "Execute request {request_id} through the inventory and order tools. "
     "Preserve its resolved recipient and destination, and prefer its requested SKU "
     "over the fallback whenever stock permits."),
    ("gx_submission", "Place a single order for request {request_id}. Derive the missing field "
     "from the supplied source records, use the given fixed fields, and substitute "
     "the fallback only when the preferred SKU has no stock."),
)
GX_DEMO_CITIES = ("Valletta", "Reykjavik", "Suva", "La Paz")
GX_SMOKE_TEMPLATE = (
    "gx_smoke",
    "Exercise the tool interface for request {request_id}: resolve its field, check inventory, "
    "reserve and submit one order using the allowed fallback if necessary.",
)
GX_ORDER_FIELDS = {
    "delivery_city": "city", "order_customer": "customer", "preferred_item": "item",
}


def gx_build_cases(stage="evaluation", seed=83021):
    """Return 192 evaluation sources or six independent usability smoke sources.

    A six-source block contains every attribute x stock combination. Four
    consecutive blocks also cover all four templates, so each 24-source chunk
    is balanced over all three factors. City pools cycle in shuffled order per
    attribute, covering every evaluation city without token eligibility filters.
    Opaque identifiers are independent random draws, never value/role suffixes.
    """
    if stage not in ("evaluation", "smoke"):
        raise ValueError("stage must be evaluation or smoke")
    if type(seed) is not int:
        raise ValueError("seed must be an integer")
    rng = gx_random.Random(f"gx-generalization-v1:{stage}:{seed}")
    used_ids = set()

    def gx_opaque(prefix):
        while True:
            value = prefix + "-" + f"{rng.getrandbits(64):016x}"
            if value not in used_ids:
                used_ids.add(value)
                return value

    cities = GX_CITIES if stage == "evaluation" else GX_DEMO_CITIES
    city_queues = {attribute: [] for attribute in GX_ATTRIBUTES}

    def gx_next_city(attribute):
        if not city_queues[attribute]:
            city_queues[attribute] = list(cities)
            rng.shuffle(city_queues[attribute])
        return city_queues[attribute].pop()

    templates = GX_TEMPLATES if stage == "evaluation" else (GX_SMOKE_TEMPLATE,)
    replicates = range(8) if stage == "evaluation" else range(1)
    cases = []
    for replicate in replicates:
        for template_id, template_text in templates:
            for attribute in GX_ATTRIBUTES:
                for stock_available in (True, False):
                    source_id = gx_opaque("gx-" + stage)
                    request_id = gx_opaque("req")
                    customer, alternative_customer = gx_opaque("acct"), gx_opaque("acct")
                    preferred, fallback, alternative_item = (
                        gx_opaque("sku"), gx_opaque("sku"), gx_opaque("sku"))
                    city = gx_next_city(attribute)
                    other_city = rng.choice([value for value in cities if value != city])
                    gold = {"delivery_city": city, "order_customer": customer,
                            "preferred_item": preferred}
                    competing = {"delivery_city": other_city,
                                 "order_customer": alternative_customer,
                                 "preferred_item": alternative_item}
                    case = {
                        "source_id": source_id, "stage": stage, "attribute": attribute,
                        "template_id": template_id, "template_text": template_text,
                        "replicate": replicate, "request_id": request_id,
                        "gold_fields": gold, "conflict_value": competing[attribute],
                        "fallback_item": fallback, "alternative_item": alternative_item,
                        "profile_event_id": gx_opaque("evt"), "note_event_id": gx_opaque("evt"),
                        "stock_available": stock_available,
                        "initial_env": {
                            "stock": {preferred: int(stock_available), fallback: 2,
                                      alternative_item: 2},
                            "reservations": {}, "orders": [],
                            "next_reservation": 1, "next_order": 1,
                        },
                    }
                    gx_validate_case(case)
                    cases.append(case)
    return cases


def gx_validate_case(case):
    """Validate evaluator metadata without exposing it to model generation."""
    if case.get("attribute") not in GX_ATTRIBUTES:
        raise ValueError("Unknown focal attribute")
    gold = case.get("gold_fields")
    if not isinstance(gold, dict) or set(gold) != set(GX_ATTRIBUTES):
        raise ValueError("gold_fields must contain the three task fields")
    if any(not isinstance(value, str) or not value.strip() for value in gold.values()):
        raise ValueError("Gold task fields must be nonempty strings")
    for key in ("source_id", "request_id", "conflict_value", "fallback_item",
                "profile_event_id", "note_event_id", "template_id", "template_text"):
        if not isinstance(case.get(key), str) or not case[key].strip():
            raise ValueError("Missing/nonstring case field: " + key)
    if case["conflict_value"] == gold[case["attribute"]]:
        raise ValueError("Conflict value must differ from the verified focal value")
    if case["request_id"] == gold["order_customer"]:
        raise ValueError("Request identity must not be the answer customer")
    if case["profile_event_id"] == case["note_event_id"]:
        raise ValueError("The two sources require distinct event IDs")
    if case["fallback_item"] == gold["preferred_item"]:
        raise ValueError("Fallback and preferred item must differ")
    stock = case["initial_env"]["stock"]
    if gold["preferred_item"] not in stock or stock.get(case["fallback_item"], 0) <= 0:
        raise ValueError("The preferred SKU must exist and the distinct fallback must be stocked")
    if type(case.get("stock_available")) is not bool or case["stock_available"] != (stock[gold["preferred_item"]] > 0):
        raise ValueError("stock_available does not match the initial environment")
    if case["attribute"] == "preferred_item":
        if case["conflict_value"] == case["fallback_item"] or stock.get(case["conflict_value"], 0) <= 0:
            raise ValueError("Conflicting preferred SKU must be a distinct stocked alternative")
    return case


def gx_memory(case, condition, position, arm):
    """Make paired records, then apply source-kind-only structural baselines."""
    gx_validate_case(case)
    if condition not in ("aligned", "conflict"):
        raise ValueError("Unknown context condition")
    if position not in ("profile_first", "note_first"):
        raise ValueError("Unknown input record position")
    if arm not in ("untreated", "authority_first", "authority_only"):
        raise ValueError("Unknown structural baseline")
    profile = {"kind": "verified_profile", "entity": case["request_id"],
               "attribute": case["attribute"],
               "value": case["gold_fields"][case["attribute"]],
               "source_event": case["profile_event_id"]}
    note = {"kind": "retrieved_note", "entity": case["request_id"],
            "attribute": case["attribute"],
            "value": profile["value"] if condition == "aligned" else case["conflict_value"],
            "source_event": case["note_event_id"]}
    records = [profile, note] if position == "profile_first" else [note, profile]
    if arm == "authority_first":
        records = ([record for record in records if record["kind"] == "verified_profile"]
                   + [record for record in records if record["kind"] != "verified_profile"])
    elif arm == "authority_only":
        records = [record for record in records if record["kind"] == "verified_profile"]
    return gx_copy.deepcopy(records)


def gx_task_payload(case):
    """Only the other two fixed fields and fallback accompany the request ID."""
    return {"request_id": case["request_id"], "resolve_attribute": case["attribute"],
            "fixed_fields": {attribute: case["gold_fields"][attribute]
                             for attribute in GX_ATTRIBUTES if attribute != case["attribute"]},
            "fallback_item": case["fallback_item"]}


def gx_actor_examples():
    """Independent tool examples cover all field mappings and real fallback.

    These short teaching cases use only demo cities and identifiers. They never
    depend on evaluation cases, and do not furnish actions for evaluated tasks.
    """
    examples = []
    for index, attribute in enumerate(GX_ATTRIBUTES):
        request = ("req-demo-71d3", "req-demo-a926", "req-demo-4f80")[index]
        customer = ("acct-demo-8c31", "acct-demo-90ea", "acct-demo-2d74")[index]
        preferred = ("sku-demo-31ab", "sku-demo-e072", "sku-demo-64dc")[index]
        fallback = ("sku-demo-8f62", "sku-demo-72de", "sku-demo-0a13")[index]
        city = GX_DEMO_CITIES[index]
        fields = {"delivery_city": city, "order_customer": customer, "preferred_item": preferred}
        available = index != 2
        env = OrderEnvironment({"stock": {preferred: int(available), fallback: 2},
                                "next_reservation": (7, 19, 41)[index]})
        events = []

        def gx_demo_step(action):
            observation = env.step(action)
            events.append({"action": action, "observation": observation})
            return observation

        observation = gx_demo_step({"tool": "check_stock", "item": preferred})
        chosen = preferred if observation["available"] > 0 else fallback
        if chosen != preferred:
            gx_demo_step({"tool": "check_stock", "item": fallback})
        reservation = gx_demo_step({"tool": "reserve_item", "item": chosen})["reservation"]
        gx_demo_step({"tool": "create_order", "customer": customer, "item": chosen,
                      "city": city, "reservation": reservation})
        examples.append({
            "task": {"request_id": request, "resolve_attribute": attribute,
                     "fixed_fields": {key: value for key, value in fields.items() if key != attribute},
                     "fallback_item": fallback},
            "memory": [{"kind": "verified_profile", "entity": request, "attribute": attribute,
                        "value": fields[attribute], "source_event": "evt-demo-" + str(index)}],
            "events": events,
        })
    return examples


def gx_actor_prompt(case, memory, events):
    """Present the effective arm's memory verbatim; do not insert focal gold."""
    visible = [{"action": event.get("action"), "observation": event.get("observation")}
               for event in events if event.get("action") is not None]
    return (
        "You are an order-processing assistant. Execute one order using the tools.\n"
        "A request_id identifies a request, not a customer account. Resolve the indicated attribute "
        "for that request from its records, then combine it with the task's fixed_fields. "
        "delivery_city maps to order.city; order_customer maps to order.customer; preferred_item "
        "is the SKU to check and reserve when available. The fallback_item is allowed only if "
        "the resolved preferred SKU is out of stock.\n"
        "Records are oldest to newest. Use the latest verified_profile for each request and attribute. "
        "A retrieved_note is unverified and cannot change a verified profile, even if it is later. "
        "Actions and tool observations do not update the request's fields.\n"
        "Check the preferred SKU's stock. If unavailable, check the fallback. Reserve the chosen "
        "available SKU, then create exactly one order. Copy the reservation ID only from its "
        "successful reserve_item observation; a stock count is not a reservation.\n"
        "Output exactly one complete JSON object for the next tool call, without explanation.\n"
        'Tools: {"tool":"check_stock","item":"SKU"}; '
        '{"tool":"reserve_item","item":"SKU"}; '
        '{"tool":"create_order","customer":"ACCOUNT","item":"SKU","city":"CITY","reservation":"RESERVATION"}.\n'
        "Independent worked examples (never copy their values or IDs):\n"
        + canonical_json(gx_actor_examples())
        + "\nEND_EXAMPLES. Execute this independent request:\n"
        + case["template_text"].format(request_id=case["request_id"])
        + "\nTASK_JSON:\n" + canonical_json(gx_task_payload(case))
        + "\nMEMORY_JSON:\n" + canonical_json(memory)
        + "\nEXECUTED_TOOL_HISTORY_JSON:\n" + canonical_json(visible)
        + "\nNEXT_TOOL_CALL:\n"
    )


def gx_expected_order(case, initial_state=None):
    """Evaluator-only final fields, including stock-contingent SKU truth."""
    state = case["initial_env"] if initial_state is None else initial_state
    fields = case["gold_fields"]
    preferred = fields["preferred_item"]
    item = preferred if state["stock"].get(preferred, 0) > 0 else case["fallback_item"]
    return {"city": fields["delivery_city"], "customer": fields["order_customer"], "item": item}


def gx_score_actor(case, events, orders, initial_state):
    """Score accepted fields and the executed prerequisite subsequence.

    A wrong focal field is counted on any accepted order, independently of the
    correctness of the other fields. Missing orders are failures and have their
    own indicator; they never masquerade as correct field decisions. Tool
    recovery is allowed: strict success requires a valid final check/reserve/
    order subsequence, while all earlier tool errors remain reported separately.
    """
    expected = gx_expected_order(case, initial_state)
    focal = GX_ORDER_FIELDS[case["attribute"]]
    correct = len(orders) == 1 and all(orders[0].get(key) == value for key, value in expected.items())
    preferred = case["gold_fields"]["preferred_item"]
    chosen = expected["item"]
    checked, grounded = {}, set()
    checked_preferred = checked_chosen = reserved_chosen = path_valid = False
    tool_errors = []
    actions = []
    first_order = None
    for step, event in enumerate(events):
        action = event.get("action")
        observation = event.get("observation")
        if isinstance(action, dict):
            actions.append(action)
            if first_order is None and action.get("tool") == "create_order":
                first_order = action
        if not isinstance(action, dict) or not isinstance(observation, dict):
            continue
        if not observation.get("ok", False):
            tool_errors.append({"step": step, "error": observation.get("error")})
            continue
        tool, item = action.get("tool"), action.get("item")
        if tool == "check_stock":
            checked[item] = observation["available"]
            checked_preferred |= item == preferred
            checked_chosen |= item == chosen and observation["available"] > 0
        elif tool == "reserve_item" and item == chosen:
            reserved_chosen = True
            priority_ok = (checked.get(preferred, -1) > 0 if chosen == preferred
                           else checked.get(preferred, -1) == 0)
            if priority_ok and checked.get(chosen, 0) > 0:
                grounded.add(observation["reservation"])
        elif tool == "create_order" and item == chosen:
            path_valid |= action.get("reservation") in grounded
    protocol_errors = {"action_must_be_object", "unknown_tool", "invalid_arguments"}
    format_failure = any(event.get("parse_error") is not None for event in events) or any(
        error["error"] in protocol_errors for error in tool_errors)
    return {
        "task_success": bool(correct), "strict_task_success": bool(correct and path_valid),
        "accepted_wrong_attribute": any(order.get(focal) != expected[focal] for order in orders),
        "accepted_conflict_value": any(order.get(focal) == case["conflict_value"] for order in orders),
        "accepted_correct_attribute": any(order.get(focal) == expected[focal] for order in orders),
        "accepted_wrong_other_fields": any(any(order.get(key) != value for key, value in expected.items()
                                                if key != focal) for order in orders),
        "no_accepted_order": not orders, "accepted_order_count": len(orders),
        "format_failure": bool(format_failure), "tool_error": bool(tool_errors),
        "tool_error_count": len(tool_errors), "action_count": len(actions),
        "order_attempted": first_order is not None,
        "first_order_attribute_correct": first_order is not None and first_order.get(focal) == expected[focal],
        "first_order_conflict_value": first_order is not None and first_order.get(focal) == case["conflict_value"],
        "prerequisite_path_valid": bool(path_valid),
        "checked_preferred": bool(checked_preferred), "checked_chosen_available": bool(checked_chosen),
        "reserved_chosen": bool(reserved_chosen),
        "subgoal_progress": (int(checked_preferred) + int(checked_chosen) + int(reserved_chosen) + int(correct)) / 4,
    }


def gx_run_actor(backend, case, memory, cfg):
    """Generate whole actions and execute the original tools without repairs."""
    gx_validate_case(case)
    cfg = dict(cfg)
    max_actions = cfg.get("max_actions", 6)
    if type(max_actions) is not int or max_actions < 1:
        raise ValueError("max_actions must be a positive integer")
    initial = gx_copy.deepcopy(case["initial_env"])
    presented = gx_copy.deepcopy(memory)
    env = OrderEnvironment(initial)
    events = []
    token_count = prompt_tokens = 0
    model_seconds = 0.0
    terminal = "action_budget"
    started = gx_time.perf_counter()
    for step in range(max_actions):
        prompt = gx_actor_prompt(case, presented, events)
        output = backend.generate(prompt, role="actor", heads=(),
                                  max_new_tokens=cfg.get("actor_max_new_tokens", 128),
                                  use_cache=cfg.get("use_cache", True))
        raw = output.get("text", "")
        action, parse_error = parse_json_output(raw, dict)
        event = {"step": step, "prompt": prompt, "raw_output": raw,
                 "action": action, "observation": None, "parse_error": parse_error,
                 "stop_reason": output.get("stop_reason")}
        token_count += int(output.get("token_count", 0))
        prompt_tokens += int(output.get("prompt_tokens", 0))
        model_seconds += float(output.get("seconds", 0.0))
        if parse_error is not None:
            events.append(event)
            terminal = "format_failure"
            break
        observation = env.step(action)
        event["observation"] = observation
        events.append(event)
        if observation.get("error") in {"action_must_be_object", "unknown_tool", "invalid_arguments"}:
            terminal = "protocol_failure"
            break
        if action.get("tool") == "create_order" and observation.get("ok"):
            terminal = "order_created"
            break
    orders = gx_copy.deepcopy(env.state["orders"][len(initial.get("orders", [])):])
    metrics = gx_score_actor(case, events, orders, initial)
    metrics["budget_exhausted"] = terminal == "action_budget"
    return {
        "source_id": case["source_id"], "stage": case["stage"],
        "attribute": case["attribute"], "request_id": case["request_id"],
        "template_id": case["template_id"], "stock_available": case["stock_available"],
        "memory": presented, "events": events, "orders": orders,
        "actions": [gx_copy.deepcopy(event["action"]) for event in events if event.get("action") is not None],
        "initial_state": initial, "final_state": env.snapshot(),
        "terminal": terminal, "metrics": metrics, **metrics,
        "model_calls": len(events), "token_count": token_count, "prompt_tokens": prompt_tokens,
        "model_seconds": model_seconds, "seconds": gx_time.perf_counter() - started,
        "evaluation_layer": "end_to_end", "heads": [],
    }

# Append-only Writer evaluation for the three-attribute generalization study.
# All extension definitions use gx_/GX_ names and leave the loaded v3 model,
# functions, configuration, datasets, interventions and run directory untouched.
import copy as gx_copy
import math as gx_math


GX_WRITER_ATTRIBUTES = ("delivery_city", "order_customer", "preferred_item")


def gx_history(case, memory, actor_result):
    """Freeze the arm's supplied records and its actually generated Actor events.

    No profile is reconstructed from case.gold_fields. In particular, filtering
    or reordering performed by an input baseline is preserved. The sequence
    index records presentation order; it is not a fabricated event timestamp.
    Wrong orders and fallback selections remain observed actions, never profile
    updates. Actor prompt text is not duplicated into the Writer history.
    """
    if not isinstance(memory, list):
        raise TypeError("gx_history requires the effective presented memory list")
    if not isinstance(actor_result, dict) or not isinstance(actor_result.get("events"), list):
        raise TypeError("gx_history requires an actual Actor result with events")
    if actor_result.get("source_id") != case["source_id"]:
        raise ValueError("Actor trajectory belongs to another source")
    if "memory" in actor_result and actor_result["memory"] != memory:
        raise ValueError("Actor trajectory used a different arm's effective memory")
    for field in ("attribute", "request_id"):
        if field in actor_result and actor_result[field] != case[field]:
            raise ValueError(f"Actor trajectory has a different {field}")
    history = []
    for record in memory:
        if not isinstance(record, dict):
            raise TypeError("Effective memory records must be dictionaries")
        event = gx_copy.deepcopy(record)
        event["sequence"] = len(history)
        history.append(event)
    for index, event in enumerate(actor_result["events"]):
        if not isinstance(event, dict):
            raise TypeError("Actor events must be dictionaries")
        origin = f'{case["source_id"]}:observed:{index}'
        if event.get("action") is not None:
            history.append({
                "kind": "observed_action", "sequence": len(history),
                "source_event": origin + ":action",
                "action": gx_copy.deepcopy(event["action"]),
            })
        elif event.get("raw_output") is not None or event.get("parse_error") is not None:
            history.append({
                "kind": "observed_output", "sequence": len(history),
                "source_event": origin + ":output",
                "text": gx_copy.deepcopy(event.get("raw_output", "")),
                "parse_error": gx_copy.deepcopy(event.get("parse_error")),
            })
        if event.get("observation") is not None:
            history.append({
                "kind": "tool_observation", "sequence": len(history),
                "source_event": origin + ":observation",
                "observation": gx_copy.deepcopy(event["observation"]),
            })
    return history


def gx_writer_reference(history):
    """Latest authorized profile per entity/attribute, never notes or executions."""
    if not isinstance(history, list):
        raise TypeError("Writer reference history must be a list")
    profiles = []
    for index, event in enumerate(history):
        if not isinstance(event, dict):
            raise TypeError("Writer history events must be dictionaries")
        if event.get("kind") != "verified_profile":
            continue
        fields = ("entity", "attribute", "value", "source_event")
        if any(not isinstance(event.get(key), str) or not event[key].strip() for key in fields):
            raise ValueError("Malformed verified profile in Writer history")
        if event["attribute"] not in GX_WRITER_ATTRIBUTES:
            raise ValueError("Unsupported verified profile attribute")
        sequence = event.get("sequence", index)
        if (isinstance(sequence, bool) or not isinstance(sequence, (int, float))
                or not gx_math.isfinite(sequence)):
            raise ValueError("Verified profile sequence must be a finite number")
        profiles.append((sequence, index, event))
    reference = {}
    for _, _, event in sorted(profiles, key=lambda item: (item[0], item[1])):
        reference[event["entity"], event["attribute"]] = {
            "kind": "preference", **{key: event[key] for key in ("entity", "attribute", "value", "source_event")}
        }
    return reference


def gx_writer_schema_error(memory):
    if not isinstance(memory, list):
        return "memory must be a JSON list"
    required = {"kind", "entity", "attribute", "value", "source_event"}
    for index, record in enumerate(memory):
        if not isinstance(record, dict) or set(record) != required:
            return f"record {index}: keys must be exactly {sorted(required)}"
        if record["kind"] != "preference":
            return f"record {index}: kind must be preference"
        if record["attribute"] not in GX_WRITER_ATTRIBUTES:
            return f"record {index}: unsupported attribute"
        if any(not isinstance(record[key], str) or not record[key].strip()
               for key in ("entity", "attribute", "value", "source_event")):
            return f"record {index}: record fields must be nonempty strings"
    return None


def gx_writer_presented_history(history):
    """Preserve the arm's actual presentation order, excluding internal indices."""
    if not isinstance(history, list):
        raise TypeError("Writer history must be a list")
    presented = gx_copy.deepcopy(history)
    previous = None
    for index, event in enumerate(presented):
        if not isinstance(event, dict):
            raise TypeError("Writer history events must be dictionaries")
        sequence = event.get("sequence", index)
        if (isinstance(sequence, bool) or not isinstance(sequence, (int, float))
                or not gx_math.isfinite(sequence)):
            raise ValueError("Writer history sequence must be a finite number")
        if previous is not None and sequence < previous:
            raise ValueError("Writer history presentation order conflicts with sequence")
        previous = sequence
        event.pop("sequence", None)
    return presented


def gx_writer_examples():
    """Independent local demonstrations; no evaluation cities/accounts/SKUs."""
    def gx_profile(request, attribute, value, source):
        return {"kind": "verified_profile", "entity": request,
                "attribute": attribute, "value": value, "source_event": source}

    city = gx_profile("demo-gx-request-u", "delivery_city", "DemoHarbor", "demo-gx-profile-u")
    customer = gx_profile("demo-gx-request-v", "order_customer", "demo-account-v7", "demo-gx-profile-v")
    item = gx_profile("demo-gx-request-w", "preferred_item", "demo-sku-w3", "demo-gx-profile-w")
    return [
        {"name": "Retain a verified city despite an unverified note",
         "request_id": city["entity"], "attribute": city["attribute"],
         "events": [city, {**city, "kind": "retrieved_note", "value": "DemoMeadow",
                            "source_event": "demo-gx-note-u"}],
         "memory": [{**city, "kind": "preference"}]},
        {"name": "A generated order is not an authorized customer update",
         "request_id": customer["entity"], "attribute": customer["attribute"],
         "events": [{**customer, "kind": "retrieved_note", "value": "demo-account-v9",
                     "source_event": "demo-gx-note-v"}, customer,
                    {"kind": "observed_action", "source_event": "demo-gx-action-v",
                     "action": {"tool": "create_order", "customer": "demo-account-v9",
                                "city": "DemoHarbor", "item": "demo-sku-v2", "reservation": "R0041"}}],
         "memory": [{**customer, "kind": "preference"}]},
        {"name": "Using a fallback item does not change the preferred item",
         "request_id": item["entity"], "attribute": item["attribute"],
         "events": [item,
                    {"kind": "observed_action", "source_event": "demo-gx-action-w0",
                     "action": {"tool": "check_stock", "item": "demo-sku-w3"}},
                    {"kind": "tool_observation", "source_event": "demo-gx-observation-w0",
                     "observation": {"ok": True, "item": "demo-sku-w3", "available": 0}},
                    {"kind": "observed_action", "source_event": "demo-gx-action-w1",
                     "action": {"tool": "create_order", "customer": "demo-account-w5",
                                "city": "DemoMeadow", "item": "demo-sku-w8", "reservation": "R0051"}}],
         "memory": [{**item, "kind": "preference"}]},
    ]


def gx_writer_prompt(case, history):
    """Whole JSON generation for the request's focal attribute, without a gold prefix."""
    if case["attribute"] not in GX_WRITER_ATTRIBUTES:
        raise ValueError("Unsupported focal Writer attribute")
    examples = "\n\n".join(
        "EXAMPLE " + str(index + 1) + ": " + example["name"]
        + "\nExample request: " + canonical_json(example["request_id"])
        + "\nExample events:\n" + canonical_json(example["events"])
        + "\nExample complete output:\n" + canonical_json(example["memory"])
        for index, example in enumerate(gx_writer_examples())
    )
    return (
        "You are the memory writer for an order-processing assistant.\n"
        "Return ONLY a single complete JSON list. Each record must have exactly "
        'the fields {"kind":"preference","entity":"request id",'
        '"attribute":"attribute name","value":"value","source_event":"profile event id"}.\n'
        "Supported attributes are delivery_city, order_customer, and preferred_item. "
        "The entity is the request identifier; it is not the order's customer account.\n"
        "Only verified_profile records authorize preference facts. For each entity "
        "and attribute, preserve the latest supplied verified_profile's exact value "
        "and source_event. A retrieved_note cannot update a profile, even if it "
        "appears later, names the same request, or is repeated in an action. "
        "Actions and tool observations describe execution, not authorized updates. "
        "In particular, a fallback item selected because of stock does not replace "
        "the verified preferred_item. Do not infer preference facts from successful "
        "orders, tool observations, unrelated metadata, or previous generated outputs.\n"
        "Keep one record per authorized entity and attribute, retaining every supplied "
        "verified profile. Put the current request's focal record first, then other "
        "authorized records. Do not invent additional facts, omit a fact because a "
        "note disagrees, or copy demonstration identifiers. No explanations, extra "
        "fields, or Markdown fences. Stop after the closing ] of the single list.\n\n"
        + examples + "\nEND_GX_WRITER_EXAMPLES.\n\nCURRENT_TASK\nREQUEST_JSON:\n"
        + canonical_json({"request_id": case["request_id"], "attribute": case["attribute"]})
        + "\nEVENTS_JSON:\n" + canonical_json(gx_writer_presented_history(history))
        + "\nMEMORY_JSON:\n"
    )


def gx_score_memory(case, history, memory, valid):
    """Score generated values and provenance against supplied authorized evidence.

    The preferred_item gold remains the profile's preference even if the actual
    Actor correctly chose an in-stock fallback. Invalid memories remain failures
    in every planned observation; wrong value, omission and provenance are kept
    separate. Whole-memory correctness rejects extra unsupported records.
    """
    reference = gx_writer_reference(history)
    valid = bool(valid) and gx_writer_schema_error(memory) is None
    records = memory if valid else []
    by_key = {}
    for record in records:
        by_key.setdefault((record["entity"], record["attribute"]), []).append(record)
    focal_key = (case["request_id"], case["attribute"])
    focal_reference = reference.get(focal_key)
    focal_records = by_key.get(focal_key, [])
    has_reference = bool(reference)
    value_correct = (valid and len(focal_records) == 1
                     and focal_records[0]["value"] == focal_reference["value"]
                     if focal_reference is not None else None)
    source_correct = (valid and len(focal_records) == 1
                      and focal_records[0]["source_event"] == focal_reference["source_event"]
                      if focal_reference is not None else None)
    conflict_error = (any(record["value"] == case["conflict_value"] for record in focal_records)
                      and case["conflict_value"] != focal_reference["value"]
                      if focal_reference is not None else None)
    missing = sum(key not in by_key for key in reference)
    unsupported = (sum(key not in reference or record["value"] != reference[key]["value"]
                       for key, entries in by_key.items() for record in entries)
                   if has_reference else None)
    source_mismatch = (sum(key in reference and record["source_event"] != reference[key]["source_event"]
                           for key, entries in by_key.items() for record in entries)
                       if has_reference else None)
    correct_count = (sum(len(by_key.get(key, [])) == 1
                         and by_key[key][0]["value"] == profile["value"]
                         for key, profile in reference.items()) if has_reference else None)
    joint_count = (sum(len(by_key.get(key, [])) == 1
                       and by_key[key][0]["value"] == profile["value"]
                       and by_key[key][0]["source_event"] == profile["source_event"]
                       for key, profile in reference.items()) if has_reference else None)
    return {
        "target_correct": value_correct, "target_source_correct": source_correct,
        "target_joint_correct": (bool(value_correct and source_correct)
                                 if focal_reference is not None else None),
        "target_conflict_error": conflict_error,
        "whole_memory_correct": (valid and len(records) == len(reference)
                                 and joint_count == len(reference) if has_reference else None),
        "format_failure": not valid,
        "omission": bool(missing) if has_reference else None,
        "target_omission": not focal_records if focal_reference is not None else None,
        "duplicate_count": sum(max(0, len(entries) - 1) for entries in by_key.values()),
        "missing_count": missing if has_reference else None,
        "unsupported_count": unsupported, "source_mismatch_count": source_mismatch,
        "required_count": len(reference), "record_count": len(records),
        "correct_count": correct_count, "joint_correct_count": joint_count,
        "reference_available": focal_reference is not None,
        "reference_policy": "gx_latest_supplied_verified_profile_v1",
    }


def gx_run_writer(backend, case, history, cfg):
    """Run the original backend with empty heads and retain complete raw output."""
    evidence = gx_copy.deepcopy(history)
    before = stable_hash(evidence)
    prompt = gx_writer_prompt(case, evidence)
    generation = backend.generate(
        prompt, role="writer", heads=(),
        max_new_tokens=cfg.get("writer_max_new_tokens", 384),
        use_cache=cfg.get("use_cache", True),
    )
    raw = generation.get("text", "")
    memory, parse_error = parse_json_output(raw, expected_type=list)
    error = parse_error or gx_writer_schema_error(memory)
    valid = error is None
    memory = memory if valid else None
    if stable_hash(history) != before or stable_hash(evidence) != before:
        raise RuntimeError("GX Writer modified its actual input history")
    return {
        "memory": memory, "valid": valid, "raw": raw, "error": error,
        "metrics": gx_score_memory(case, evidence, memory, valid),
        "history_hash": before, "prompt_hash": stable_hash(prompt),
        "token_count": generation.get("token_count", 0),
        "prompt_tokens": generation.get("prompt_tokens", 0),
        "seconds": generation.get("seconds", 0.0),
        "stop_reason": generation.get("stop_reason"),
        "model_calls": 1, "heads": [],
        "memory_origin": "gx_model_generated_from_effective_input_and_actual_actor_trace",
    }

"""Isolated, resumable continuation of conflict-v3; no model loading or training."""
import copy
import csv
import datetime
import itertools
import json
import math
import marshal
from pathlib import Path
import random
import zipfile


GX_ARMS = ("untreated", "authority_first", "authority_only")
GX_CONDITIONS = ("aligned", "conflict")
GX_POSITIONS = ("profile_first", "note_first")
GX_PRIMARY_METRICS = ("actor_accepted_wrong_attribute", "actor_task_success",
                      "writer_target_joint_correct", "writer_target_conflict_error")


def gx_code_signature():
    """Detect redefining live extension functions/constants after the run is locked."""
    definitions = {}
    for name, value in list(globals().items()):
        if not name.startswith(("gx_", "GX")):
            continue
        if hasattr(value, "__code__"):
            # Version 0 avoids reference/interning flags that can change after
            # a code object is first executed despite identical executable code.
            definitions[name] = marshal.dumps(value.__code__, 0).hex()
        elif isinstance(value, type):
            definitions[name] = {key: marshal.dumps(method.__code__, 0).hex()
                                 for key, method in vars(value).items() if hasattr(method, "__code__")}
        elif isinstance(value, (str, bool, int, float, list, tuple, dict)):
            definitions[name] = value
    return stable_hash(definitions)


def gx_wilson(successes, total):
    if not total:
        return [None, None]
    z = 1.959963984540054
    p = successes / total
    divisor = 1 + z * z / total
    center = (p + z * z / (2 * total)) / divisor
    radius = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / divisor
    return [max(0.0, center - radius), min(1.0, center + radius)]


def gx_pair_stats(before, after, seed=83021, reps=2000):
    """One paired difference per source; exact binary discordance test."""
    if set(before) != set(after) or not before:
        raise ValueError("Paired source coverage differs or is empty")
    sources = sorted(before)
    if any(before[s] not in (0, 1, False, True) or after[s] not in (0, 1, False, True)
           for s in sources):
        raise ValueError("Paired metric must be binary")
    differences = [int(after[s]) - int(before[s]) for s in sources]
    n = len(sources)
    plus, minus = differences.count(1), differences.count(-1)
    discordant = plus + minus
    exact = (min(1.0, 2 * sum(math.comb(discordant, k)
                             for k in range(min(plus, minus) + 1)) / 2 ** discordant)
             if discordant else 1.0)
    degenerate = len(set(differences)) == 1
    if degenerate:
        ci = [sum(differences) / n] * 2
    else:
        rng = random.Random(seed)
        draws = sorted(sum(rng.choices(differences, k=n)) / n for _ in range(reps))
        ci = [draws[int(.025 * (reps - 1))], draws[int(.975 * (reps - 1))]]
    return {"n_sources": n, "before_count": sum(before.values()),
            "after_count": sum(after.values()), "difference_after_minus_before": sum(differences) / n,
            "bootstrap_ci_low": ci[0], "bootstrap_ci_high": ci[1],
            "bootstrap_degenerate": degenerate,
            "ci_note": ("Degenerate empirical bootstrap; not evidence of population equivalence."
                        if degenerate else "Source-paired percentile bootstrap; descriptive, unadjusted."),
            "discordant_0_to_1": plus, "discordant_1_to_0": minus,
            "exact_discordance_p_unadjusted": exact}


def gx_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = sorted(set().union(*(row.keys() for row in rows))) if rows else ["no_completed_rows"]
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: canonical_json(value) if isinstance(value, (dict, list, tuple)) else value
                             for key, value in row.items()})
    temporary.replace(path)


def gx_read_checked(path, signature):
    record = json.loads(Path(path).read_text(encoding="utf-8"))
    if record.get("signature") != signature:
        raise RuntimeError(f"Stale extension checkpoint: {path}")
    if stable_hash(record.get("result")) != record.get("result_hash"):
        raise RuntimeError(f"Corrupt extension checkpoint: {path}")
    return record["result"]


def gx_write_checked(path, signature, result):
    atomic_json(path, {"signature": signature, "result_hash": stable_hash(result), "result": result})


class GXCachedBackend:
    """Content-addressed greedy generations; display-arm names never enter prompts."""
    def __init__(self, backend, directory, identity_hash):
        self.backend = backend
        self.directory = Path(directory)
        self.identity_hash = identity_hash
        self.new_calls = 0
        self.cache_hits = 0

    def generate(self, prompt, role="actor", heads=(), max_new_tokens=None, use_cache=True):
        if heads:
            raise ValueError("This extension evaluates unablated baselines only")
        payload = {"identity": self.identity_hash, "prompt": prompt, "role": role,
                   "heads": [], "max_new_tokens": max_new_tokens, "use_cache": use_cache}
        signature = stable_hash(payload)
        path = self.directory / (signature + ".json")
        if path.exists():
            self.cache_hits += 1
            return copy.deepcopy(gx_read_checked(path, signature))
        result = self.backend.generate(prompt, role=role, heads=(),
                                       max_new_tokens=max_new_tokens, use_cache=use_cache)
        gx_write_checked(path, signature, result)
        self.new_calls += 1
        return copy.deepcopy(result)


class GXExperiment:
    """Keep the parent's notebook state and all old checkpoints untouched."""
    def __init__(self, namespace, source_bundle, root_dir=None, cfg=None):
        required = ("backend", "RUN_DIR", "RUN_FINGERPRINT", "IMPLEMENTATION_HASH",
                    "MODEL_IDENTITY", "CFG", "verify_live_run")
        missing = [key for key in required if key not in namespace]
        if missing:
            raise RuntimeError(" Please stay where you are.  v3 runtime  Finish Model / Functions / Initialization of identity; missing: " + ", ".join(missing))
        namespace["verify_live_run"]()
        self.parent = namespace
        self.backend = namespace["backend"]
        if not hasattr(self.backend, "cfg") or stable_hash(self.backend.cfg) != stable_hash(namespace["CFG"]):
            raise RuntimeError("backend.cfg  And Hara  CFG  mismatch; restore the original model configuration before resuming the original experiment. ")
        self.original_dir = Path(namespace["RUN_DIR"])
        self.root = (Path(root_dir) if root_dir is not None else
                     self.original_dir.parent / (self.original_dir.name + "_generalization_v1"))
        if self.root.resolve() == self.original_dir.resolve() or self.root.resolve().is_relative_to(self.original_dir.resolve()):
            raise ValueError(" Extension directory must be located in the original  RUN_DIR  Outside ")
        self.cfg = {"actor_max_new_tokens": namespace["CFG"].get("actor_max_new_tokens", 128),
                    "writer_max_new_tokens": namespace["CFG"].get("writer_max_new_tokens", 384),
                    "max_actions": 6, "use_cache": namespace["CFG"].get("use_cache", True),
                    "bootstrap_reps": 2000}
        if cfg:
            unknown = set(cfg) - set(self.cfg)
            if unknown:
                raise ValueError(f"Unknown extension configuration: {unknown}")
            self.cfg.update(cfg)
        for key in ("actor_max_new_tokens", "writer_max_new_tokens", "max_actions", "bootstrap_reps"):
            if type(self.cfg[key]) is not int or self.cfg[key] < (100 if key == "bootstrap_reps" else 1):
                raise ValueError(f"Invalid {key}")
        if type(self.cfg["use_cache"]) is not bool:
            raise ValueError("use_cache must be boolean")
        self.cases = {stage: gx_build_cases(stage=stage, seed=83021) for stage in ("smoke", "evaluation")}
        self.protocol = {
            "version": "conflict_v3_generalization_v1", "seed": 83021,
            "source_counts": {k: len(v) for k, v in self.cases.items()},
            "attributes": list(GX_ATTRIBUTES), "cities": list(GX_CITIES),
            "conditions": list(GX_CONDITIONS), "positions": list(GX_POSITIONS), "arms": list(GX_ARMS),
            "unit": "source_id; never a condition/position/baseline row",
            "model_intervention": "none", "layer": "end_to_end",
            "baseline_policy": "kind-based preprocessing applied to Actor and its Writer history",
            "writer_truth": "latest verified focal preference, including preferred SKU when fallback is ordered",
            "primary_contrast": "conflict minus aligned, untreated, note_first, accepted_wrong_attribute",
            "secondary_contrasts": "source order and simple baselines; stratified attributes; unadjusted",
            "generalization_scope": "new experimental vocabulary and wording, same local order environment",
            "interpretation": "joint extension, no separate causal attribution to vocabulary vs template changes",
            "source_tags_assumed_reliable": True, "old_test_used": False,
            "head_search_used": False, "error_based_selection": False,
            "frozen_before_generation": True, "config": copy.deepcopy(self.cfg),
        }
        self.identity = {
            "protocol": copy.deepcopy(self.protocol), "extension_code_hash": stable_hash(source_bundle),
            "live_extension_signature": gx_code_signature(),
            "original_run_fingerprint": namespace["RUN_FINGERPRINT"],
            "original_implementation_hash": namespace["IMPLEMENTATION_HASH"],
            "original_config": copy.deepcopy(namespace["CFG"]),
            "model": copy.deepcopy(namespace["MODEL_IDENTITY"]),
            "backend_config": copy.deepcopy(getattr(self.backend, "cfg", {})),
            "manifest_hashes": {k: stable_hash(v) for k, v in self.cases.items()},
        }
        self.identity_hash = stable_hash(self.identity)
        self.root.mkdir(parents=True, exist_ok=True)
        identity_file = self.root / "identity.json"
        if identity_file.exists():
            stored = json.loads(identity_file.read_text(encoding="utf-8"))
            if stored != self.identity:
                raise RuntimeError(" Extended version / Agreement / The model has changed; please use a new extension directory and not mix the old results. ")
        else:
            atomic_json(identity_file, self.identity)
        for stage, cases in self.cases.items():
            path = self.root / (stage + "-manifest.json")
            if path.exists() and json.loads(path.read_text(encoding="utf-8")) != cases:
                raise RuntimeError(f"Frozen manifest changed: {path}")
            if not path.exists():
                atomic_json(path, cases)
        self.cached_backend = GXCachedBackend(self.backend, self.root / "items" / "generations", self.identity_hash)

    def _ensure_live(self, check_disk=True):
        self.parent["verify_live_run"]()
        if stable_hash(self.identity) != self.identity_hash:
            raise RuntimeError("Frozen extension identity was modified")
        if gx_code_signature() != self.identity["live_extension_signature"]:
            raise RuntimeError("Live extension functions/constants changed after protocol lock")
        if self.parent["backend"] is not self.backend:
            raise RuntimeError("Loaded backend was replaced; recreate extension after verifying the original runtime")
        checks = {"original_run_fingerprint": self.parent["RUN_FINGERPRINT"],
                  "original_implementation_hash": self.parent["IMPLEMENTATION_HASH"],
                  "original_config": self.parent["CFG"], "model": self.parent["MODEL_IDENTITY"],
                  "backend_config": getattr(self.backend, "cfg", {})}
        if any(stable_hash(value) != stable_hash(self.identity[key]) for key, value in checks.items()):
            raise RuntimeError("Original model/runtime/config identity changed")
        if Path(self.parent["RUN_DIR"]) != self.original_dir:
            raise RuntimeError("Original RUN_DIR changed; do not overlap extension with a directory-switching cell")
        if stable_hash(self.cfg) != stable_hash(self.identity["protocol"]["config"]):
            raise RuntimeError("Extension config changed after manifest lock")
        for stage, cases in self.cases.items():
            if stable_hash(cases) != self.identity["manifest_hashes"][stage]:
                raise RuntimeError("In-memory frozen manifest changed")
        if check_disk:
            if json.loads((self.root / "identity.json").read_text(encoding="utf-8")) != self.identity:
                raise RuntimeError("Frozen identity on disk changed")
            for stage, cases in self.cases.items():
                if json.loads((self.root / (stage + "-manifest.json")).read_text(encoding="utf-8")) != cases:
                    raise RuntimeError("Frozen manifest on disk changed")

    def _path(self, case):
        return self.root / "sources" / case["stage"] / (case["source_id"] + ".json")

    def _signature(self, case):
        return stable_hash({"identity": self.identity_hash, "case": case})

    def _read_source(self, case):
        result = gx_read_checked(self._path(case), self._signature(case))
        expected = set(itertools.product(GX_ARMS, GX_CONDITIONS, GX_POSITIONS))
        keys = [(r["arm"], r["condition"], r["position"]) for r in result["rows"]]
        if len(keys) != len(expected) or set(keys) != expected:
            raise RuntimeError("Completed source has missing or duplicate paired rows")
        if any(r["source_id"] != case["source_id"] for r in result["rows"]):
            raise RuntimeError("Source checkpoint provenance mismatch")
        return result

    def _run_source(self, case):
        rows, episodes = [], {}
        # Order is fixed before observing outcomes. Identical inputs share inference.
        for arm, condition, position in itertools.product(GX_ARMS, GX_CONDITIONS, GX_POSITIONS):
            memory = gx_memory(case, condition, position, arm)
            episode_id = stable_hash({"case": case, "effective_memory": memory})
            if episode_id not in episodes:
                actor = gx_run_actor(self.cached_backend, copy.deepcopy(case), copy.deepcopy(memory), dict(self.cfg))
                history = gx_history(case, memory, actor)
                writer = gx_run_writer(self.cached_backend, copy.deepcopy(case), history, dict(self.cfg))
                episodes[episode_id] = {"memory": memory, "actor": actor, "writer": writer,
                                        "history_hash": stable_hash(history)}
            episode = episodes[episode_id]
            row = {"source_id": case["source_id"], "stage": case["stage"],
                   "attribute": case["attribute"], "template_id": case["template_id"],
                   "stock_available": case["stock_available"],
                   "city": case["gold_fields"]["delivery_city"],
                   "multiword_city": " " in case["gold_fields"]["delivery_city"],
                   "arm": arm, "condition": condition, "position": position,
                   "episode_id": episode_id,
                   **{"actor_" + k: v for k, v in episode["actor"]["metrics"].items()},
                   **{"writer_" + k: v for k, v in episode["writer"]["metrics"].items()}}
            rows.append(row)
        if len(episodes) != 5:
            raise RuntimeError(f"Expected five distinct effective inputs; found {len(episodes)}")
        result = {"source_id": case["source_id"], "rows": rows, "episodes": episodes}
        gx_write_checked(self._path(case), self._signature(case), result)
        return result

    def inspect(self):
        self._ensure_live()
        counts = {}
        for stage, cases in self.cases.items():
            counts[stage] = {"planned_sources": len(cases),
                             "completed_sources": sum(self._path(c).exists() for c in cases),
                             "logical_rows": len(cases) * 12,
                             "unique_actor_episodes": len(cases) * 5,
                             "unique_writer_episodes": len(cases) * 5}
        print(" Continue to reuse current models and tools; expanded directory: ", self.root)
        print(json.dumps(counts, ensure_ascii=False, indent=2))
        print("960  Complete  Actor episode  Not equal to  960  Subgeneration: Models are required for each step of tool deployment, and there are others  Writer。")
        print(" Each batch 24 Independent. source； complete evaluation as 192。smoke Do not pick the wrong question, do not trigger the wrong rate threshold. ")
        return counts

    def audit_token_budget(self):
        """Tokenizer-only legal-path length check; no model call or scored episode.

        Constructed traces here are length fixtures, never written to evaluation
        sources or passed to run_writer. Actual evaluation always uses generated
        actions. The backend still checks every actual prompt without truncation.
        """
        self._ensure_live()
        if not callable(getattr(self.backend, "encode", None)):
            print("No tokenizer encode interface; actual backend remains responsible for context limits.")
            return {"status": "unavailable"}
        configured = int(getattr(self.backend, "cfg", {}).get("max_prompt_tokens", 3072))
        model_context = getattr(getattr(getattr(self.backend, "model", None), "cfg", None), "n_ctx", None)
        limit = min(configured, int(model_context)) if model_context else configured
        maxima = {"actor": 0, "writer": 0}
        for case in self.cases["smoke"] + self.cases["evaluation"]:
            memory = gx_memory(case, "conflict", "note_first", "untreated")
            env = OrderEnvironment(case["initial_env"])
            events = []

            def execute_for_length(action):
                observation = env.step(action)
                events.append({"action": action, "observation": observation})
                return observation

            preferred = case["gold_fields"]["preferred_item"]
            preferred_observation = execute_for_length({"tool": "check_stock", "item": preferred})
            item = preferred
            if preferred_observation["available"] == 0:
                item = case["fallback_item"]
                execute_for_length({"tool": "check_stock", "item": item})
            reservation = execute_for_length({"tool": "reserve_item", "item": item})["reservation"]
            for prefix_length in range(len(events) + 1):
                maxima["actor"] = max(maxima["actor"], len(self.backend.encode(
                    gx_actor_prompt(case, memory, events[:prefix_length]))))
            execute_for_length({"tool": "create_order", "customer": case["gold_fields"]["order_customer"],
                                "city": case["gold_fields"]["delivery_city"], "item": item,
                                "reservation": reservation})
            fixture = {"source_id": case["source_id"], "events": events, "memory": memory}
            maxima["writer"] = max(maxima["writer"], len(self.backend.encode(
                gx_writer_prompt(case, gx_history(case, memory, fixture)))))
        audit = {"status": "tokenizer_only", "max_legal_path_prompt_tokens": maxima,
                 "prompt_limit": limit, "model_context": int(model_context) if model_context else None,
                 "note": "Constructed legal-path fixtures, no inference or evaluation outcome; recovery traces may be longer."}
        atomic_json(self.root / "token-budget-audit.json", audit)
        for role, tokens in maxima.items():
            if tokens > limit or (model_context and tokens + self.cfg[role + "_max_new_tokens"] > model_context):
                raise RuntimeError(f"{role} planned prompt exceeds original context budget: {audit}; shorten extension prompts in a new version.")
        print(" Only code check (no model generation): ", maxima, "； Original prompt Upper limit: ", limit)
        return audit

    def run(self, stage="evaluation", max_sources=24):
        self._ensure_live()
        if stage not in self.cases:
            raise ValueError("stage must be smoke or evaluation")
        if max_sources is not None and (type(max_sources) is not int or max_sources < 1):
            raise ValueError("max_sources must be positive integer or None (all remaining)")
        cases = self.cases[stage]
        pending = []
        for case in cases:
            if self._path(case).exists():
                self._read_source(case)
            else:
                pending.append(case)
        todo = pending if max_sources is None else pending[:max_sources]
        completed = len(cases) - len(pending)
        start_calls = self.cached_backend.new_calls
        for case in todo:
            self._ensure_live(check_disk=False)
            self._run_source(case)
            completed += 1
            print(f"[{stage}] {completed}/{len(cases)} sources complete | {case['source_id']} | "
                  f"new model calls this batch={self.cached_backend.new_calls - start_calls}", flush=True)
        report = self.report(stage)
        print(f" Add to this instalment  {len(todo)} sources； Completed  {report['completed_sources']}/{report['planned_sources']}。")
        print(" Not available source Save. Repeat this book. cell Hell run. max_sources=None Will complete all remaining tasks. ")
        return report

    def _collect(self, stage):
        if stage not in self.cases:
            raise ValueError("Unknown stage")
        complete = []
        for case in self.cases[stage]:
            if self._path(case).exists():
                complete.append(self._read_source(case))
        return complete, [row for result in complete for row in result["rows"]]

    def report(self, stage="evaluation"):
        self._ensure_live()
        complete, rows = self._collect(stage)
        metrics = GX_PRIMARY_METRICS + ("actor_strict_task_success", "actor_accepted_conflict_value",
                   "actor_format_failure", "actor_tool_error", "actor_no_accepted_order",
                   "writer_target_correct", "writer_target_source_correct", "writer_format_failure",
                   "writer_whole_memory_correct", "writer_omission")
        summaries = []
        strata = [("all", None)] + [("attribute", a) for a in GX_ATTRIBUTES]
        strata += [("stock_available", v) for v in (0, 1)]
        strata += [("template_id", v) for v in sorted({c["template_id"] for c in self.cases[stage]})]
        strata += [("city", v) for v in sorted({c["gold_fields"]["delivery_city"] for c in self.cases[stage]})]
        strata += [("multiword_city", v) for v in (False, True)]
        for stratum, value in strata:
            selected = rows if stratum == "all" else [r for r in rows if r[stratum] == value]
            for arm, condition, position in itertools.product(GX_ARMS, GX_CONDITIONS, GX_POSITIONS):
                group = [r for r in selected if (r["arm"], r["condition"], r["position"]) == (arm, condition, position)]
                if not group:
                    continue
                if len({r["source_id"] for r in group}) != len(group):
                    raise RuntimeError("Summary double-counts a source")
                for metric in metrics:
                    if metric not in group[0]:
                        continue
                    if any(r.get(metric) not in (True, False, 0, 1) for r in group):
                        raise ValueError(f"Missing/nonbinary primary metric {metric}")
                    count, n = sum(int(r[metric]) for r in group), len(group)
                    low, high = gx_wilson(count, n)
                    summaries.append({"stratum": stratum, "stratum_value": value,
                                      "arm": arm, "condition": condition, "position": position,
                                      "metric": metric, "count": count, "n_sources": n,
                                      "rate": count / n, "wilson_low": low, "wilson_high": high})
        contrasts = []
        definitions = []
        for arm, position in itertools.product(GX_ARMS, GX_POSITIONS):
            definitions.append(("conflict_minus_aligned", (arm, "aligned", position), (arm, "conflict", position)))
        for arm, condition, position in itertools.product(GX_ARMS[1:], GX_CONDITIONS, GX_POSITIONS):
            definitions.append(("baseline_minus_untreated", ("untreated", condition, position), (arm, condition, position)))
        for condition in GX_CONDITIONS:
            definitions.append(("note_first_minus_profile_first", ("untreated", condition, "profile_first"),
                                ("untreated", condition, "note_first")))
        for attribute in ("all",) + tuple(GX_ATTRIBUTES):
            selected = rows if attribute == "all" else [r for r in rows if r["attribute"] == attribute]
            for label, before_key, after_key in definitions:
                before_rows = [r for r in selected if (r["arm"], r["condition"], r["position"]) == before_key]
                after_rows = [r for r in selected if (r["arm"], r["condition"], r["position"]) == after_key]
                if not before_rows and not after_rows:
                    continue
                for metric in GX_PRIMARY_METRICS:
                    before = {r["source_id"]: r[metric] for r in before_rows}
                    after = {r["source_id"]: r[metric] for r in after_rows}
                    if len(before) != len(before_rows) or len(after) != len(after_rows):
                        raise RuntimeError("Duplicate source in paired contrast")
                    stats = gx_pair_stats(before, after, reps=self.cfg["bootstrap_reps"])
                    contrasts.append({"attribute": attribute, "contrast": label,
                                      "before": list(before_key), "after": list(after_key),
                                      "metric": metric,
                                      "analysis_role": "primary" if (
                                          attribute == "all" and metric == "actor_accepted_wrong_attribute"
                                          and label == "conflict_minus_aligned"
                                          and before_key == ("untreated", "aligned", "note_first")) else "secondary",
                                      **stats})
        failures = []
        for result in complete:
            for episode_id, episode in result["episodes"].items():
                actor, writer = episode["actor"]["metrics"], episode["writer"]["metrics"]
                if not actor["task_success"] or not writer["target_joint_correct"]:
                    failures.append({"source_id": result["source_id"], "episode_id": episode_id,
                                     "logical_arms": [{k: r[k] for k in ("arm", "condition", "position")}
                                                      for r in result["rows"] if r["episode_id"] == episode_id],
                                     **episode})
        report = {"stage": stage, "identity_hash": self.identity_hash,
                  "planned_sources": len(self.cases[stage]), "completed_sources": len(complete),
                  "complete": len(complete) == len(self.cases[stage]), "logical_rows": len(rows),
                  "unique_actor_episodes": sum(len(r["episodes"]) for r in complete),
                  "unique_writer_episodes": sum(len(r["episodes"]) for r in complete),
                  "generation_records_whole_extension_including_smoke_and_partial":
                      sum(1 for _ in (self.root / "items" / "generations").glob("*.json")),
                  "generated_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                  "analysis_status": "complete fixed manifest" if len(complete) == len(self.cases[stage]) else "PARTIAL; not full evaluation",
                  "inference_note": "Finite designed task suite; intervals reflect source resampling, not all real-world agents. Secondary p values unadjusted.",
                  "summary": summaries, "paired_effects": contrasts}
        destination = self.root / "reports" / stage
        atomic_json(destination / "report.json", report)
        atomic_json(destination / "failures.json", failures)
        gx_csv(destination / "rows.csv", rows)
        gx_csv(destination / "summary.csv", summaries)
        gx_csv(destination / "paired-effects.csv", contrasts)
        # Small on-screen summary; all attribute/template/stock/city strata stay in CSV.
        print(f"{stage}: {len(complete)}/{len(self.cases[stage])} sources; "
              f"{len(rows)} logical rows; {report['unique_actor_episodes']} distinct Actor episodes")
        for row in summaries:
            visible_metrics = GX_PRIMARY_METRICS[:3] + (("actor_format_failure", "actor_tool_error", "writer_format_failure") if stage == "smoke" else ())
            if row["stratum"] == "all" and row["condition"] == "conflict" and row["position"] == "note_first" and row["metric"] in visible_metrics:
                print(f"  {row['arm']:16s} {row['metric']:35s} {row['count']}/{row['n_sources']} ({row['rate']:.1%})")
        return report

    def export(self, stage="evaluation", download=False):
        self.report(stage)
        destination = self.root.parent / (self.root.name + "_" + stage + ".zip")
        files = [self.root / "identity.json", self.root / (stage + "-manifest.json")]
        if (self.root / "token-budget-audit.json").exists():
            files.append(self.root / "token-budget-audit.json")
        files += sorted((self.root / "reports" / stage).glob("*"))
        files += sorted((self.root / "sources" / stage).glob("*.json"))
        with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for path in files:
                if path.is_file():
                    archive.write(path, path.relative_to(self.root).as_posix())
        print(" Results package: ", destination)
        if download:
            from google.colab import files as gx_colab_files
            gx_colab_files.download(str(destination))
        return str(destination)
'''
GX_EXTENSION_NAMESPACE = {'__name__': 'gx_generalization_v1'}
for gx_required_helper in ('OrderEnvironment', 'parse_json_output', 'canonical_json', 'stable_hash', 'atomic_json'):
    if gx_required_helper not in globals():
        raise RuntimeError('Please use the original initialized v3 runtime; missing ' + gx_required_helper)
    GX_EXTENSION_NAMESPACE[gx_required_helper] = globals()[gx_required_helper]
exec(compile(GX_EXTENSION_SOURCE, '<gx-generalization-v1>', 'exec'), GX_EXTENSION_NAMESPACE)
gx = GX_EXTENSION_NAMESPACE['GXExperiment'](globals(), GX_EXTENSION_SOURCE)
GX_STATUS = gx.inspect()
GX_TOKEN_AUDIT = gx.audit_token_budget()


## 2 ·  Six independents.  smoke source
 Check only whether the new format executes normally, without selecting samples by error rate. If changes are needed  prompt， A new version has to be used. / Contents. 


In [ ]:
# BLOCK 2: six separate usability cases, never used to choose evaluation failures.
GX_SMOKE = gx.run(stage='smoke', max_sources=None)
# Inspect printed format/tool failure and success metrics. No error-rate gate.


## 3 ·  Assessment of new missions 
 Default Every  24  units, with a fixed order covering every attribute, template, and stock condition; when full  192  This will be followed by a full explanation. 


In [ ]:
# BLOCK 3: run the next 24 unfinished evaluation sources. Repeat this cell 8 times in total.
# To finish every remaining source in one run, change 24 to None.
GX_EVALUATION = gx.run(stage='evaluation', max_sources=24)


## 4 ·  Packing Export 
 Include protocol, complete new  manifest、 actual trajectories, row-level scores, statistics, failure cases, and coverage; partial export is also supported. 


In [ ]:
# BLOCK 4: download results so far; partial coverage is explicitly marked.
GX_RESULTS_ZIP = gx.export(stage='evaluation', download=True)
# Optional separate smoke export: gx.export(stage='smoke', download=True)
